In [1]:
# Importación de librerías y módulos necesarios

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

In [2]:
# Carga y preparación de datos

df_train = pd.read_csv('../../Datos/df_voices_train.csv')

# Creación de variable objetivo

df_train['y'] = df_train['Key'].map({
    'bonafide': 0,
    'spoof': 1
})

# Separación de X e y

drop_columns = ['Key', 'file_name', 'User_ID', 'Spoofing_ID', 'y']

X = df_train.drop(columns=drop_columns, errors='ignore')
y = df_train['y']

print(f"Dataset cargado: {X.shape[0]} muestras y {X.shape[1]} variables\n")

Dataset cargado: 25380 muestras y 22 variables



In [3]:
# Configuración de cross-validation

# StratifiedKFold mantiene el balance de clases en cada fold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

spoofing_ids = df_train['Spoofing_ID']

# Definición de métricas a evaluar

scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score)
}

In [4]:
# Definición de modelos

models = {

    # Random Forest

    'Random Forest (50 árboles)': RandomForestClassifier(
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    ),

    'Random Forest (100 árboles)': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    'Random Forest (150 árboles)': RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ),

    'Random Forest (200 árboles)': RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    'Random Forest (250 árboles / max_depth=5)': RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        max_depth=5,
        n_jobs=-1
    ),
    'Random Forest (250 árboles / max_depth=10)': RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        max_depth=10,
        n_jobs=-1
    ),
    'Random Forest (250 árboles / max_depth=20)': RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        max_depth=20,
        n_jobs=-1
    ),
    'Random Forest (250 árboles / max_depth=30)': RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        max_depth=30,
        n_jobs=-1
    ),
    'Random Forest (300 árboles)': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    # ---------------- GRADIENT BOOSTING ----------------
    'Gradient Boosting (50 árboles)': GradientBoostingClassifier(
        n_estimators=50,
        random_state=42
    ),

    'Random Forest (300 árboles)': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    # Gradient Boosting

    'Gradient Boosting (50 árboles)': GradientBoostingClassifier(
        n_estimators=50,
        random_state=42
    ),

    'Gradient Boosting (100 árboles)': GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ),

        'Gradient Boosting (250 árboles)': GradientBoostingClassifier(
        n_estimators=250,
        random_state=42
    ),

    # Support Vector Machine (SVM)

    'SVM (linear kernel)': SVC(
        kernel='linear',
        random_state=42
    ),

    'SVM (rbf kernel)': SVC(
        kernel='rbf',
        random_state=42
    )
}

In [5]:
# Evaluación de modelos con cross-validation

results = []

for model_name, model in models.items():

    # Pipeline:
    # 1. Escalado
    # 2. Undersampling solo en train de cada fold
    # 3. Modelo

    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('undersampling', RandomUnderSampler(random_state=42)),
        ('model', model)
    ])

    # Cross-validation

    cv_results = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        groups=spoofing_ids,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    # Promedios

    accuracy_mean = np.mean(cv_results['test_accuracy'])
    precision_mean = np.mean(cv_results['test_precision'])
    recall_mean = np.mean(cv_results['test_recall'])
    f1_mean = np.mean(cv_results['test_f1'])

    # Desviación estándar

    accuracy_std = np.std(cv_results['test_accuracy'])
    f1_std = np.std(cv_results['test_f1'])

    results.append({
        'Modelo': model_name,
        'Accuracy': accuracy_mean,
        'Precision': precision_mean,
        'Recall': recall_mean,
        'F1-Score': f1_mean,
        'Std Accuracy': accuracy_std,
        'Std F1': f1_std
    })

    print(f"Accuracy : {accuracy_mean:.4f} (+/- {accuracy_std:.4f})")
    print(f"Precision: {precision_mean:.4f}")
    print(f"Recall   : {recall_mean:.4f}")
    print(f"F1-Score : {f1_mean:.4f}")
    print()

c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Accuracy : 0.8086 (+/- 0.0077)
Precision: 0.9817
Recall   : 0.8018
F1-Score : 0.8827



c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Accuracy : 0.8127 (+/- 0.0074)
Precision: 0.9820
Recall   : 0.8063
F1-Score : 0.8855



c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


...

Accuracy : 0.8102 (+/- 0.0105)
Precision: 0.9840
Recall   : 0.8018
F1-Score : 0.8836



c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Accuracy : 0.7681 (+/- 0.0068)
Precision: 0.9788
Recall   : 0.7582
F1-Score : 0.8545



c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Accuracy : 0.7784 (+/- 0.0091)
Precision: 0.9864
Recall   : 0.7639
F1-Score : 0.8609



In [6]:
# Comparación de modelos

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='F1-Score',
    ascending=False
)

print("Resultados finales")

print(results_df.to_string(index=False))
print()

Resultados finales
                                    Modelo  Accuracy  Precision   Recall  F1-Score  Std Accuracy   Std F1
Random Forest (250 árboles / max_depth=30)  0.816588   0.981603 0.811053  0.888154      0.010080 0.006997
               Random Forest (200 árboles)  0.815879   0.981640 0.810219  0.887671      0.009434 0.006587
               Random Forest (300 árboles)  0.815524   0.981220 0.810175  0.887472      0.009760 0.006804
               Random Forest (150 árboles)  0.814106   0.981855 0.808026  0.886451      0.007926 0.005601
               Random Forest (100 árboles)  0.812687   0.982013 0.806272  0.885472      0.007433 0.005179
Random Forest (250 árboles / max_depth=20)  0.812017   0.981181 0.806228  0.885070      0.011120 0.007745
           Gradient Boosting (250 árboles)  0.810244   0.983979 0.801842  0.883561      0.010469 0.007340
                Random Forest (50 árboles)  0.808550   0.981709 0.801842  0.882666      0.007652 0.005391
           Gradient Boostin